In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Paths to your exact subfolders
TRAIN_DIR = 'dataset/seg_train'  
TEST_DIR = 'dataset/seg_test'    
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# 1. Load Datasets using modern, robust built-in utilities
print("Loading training dataset...")
train_generator = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

print("Loading test/validation dataset...")
val_generator = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 2. Rescale pixel values on the fly using a layer instead of ImageDataGenerator
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_generator = train_generator.map(lambda x, y: (normalization_layer(x), y))
val_generator = val_generator.map(lambda x, y: (normalization_layer(x), y))

# Get the list of class names to save to labels.txt later
class_names = train_generator.element_spec[0].shape # alternative below

# To explicitly extract string labels from directory:
import pathlib
data_dir = pathlib.Path(TRAIN_DIR)
extracted_labels = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
num_classes = len(extracted_labels)

# 3. Fine-tune Pre-trained Model (Same as before)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
base_model.trainable = False 

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 4. Train
print("Starting training...")
model.fit(train_generator, validation_data=val_generator, epochs=3)

# 5. Evaluate and Save
loss, accuracy = model.evaluate(val_generator)
print(f"\n--- Evaluation Metrics ---")
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

model.save('best_model.h5')
with open('labels.txt', 'w') as f:
    for label in extracted_labels:
        f.write(f"{label}\n")
print("Model saved perfectly!")

Loading training dataset...
Found 14034 files belonging to 6 classes.
Loading test/validation dataset...
Found 3000 files belonging to 6 classes.


C:\Users\shazm\AppData\Local\Temp\ipykernel_15300\2151208828.py:45: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Starting training...
Epoch 1/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 219s 485ms/step - accuracy: 0.8730 - loss: 0.3555 - val_accuracy: 0.8967 - val_loss: 0.2763
Epoch 2/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 216s 380ms/step - accuracy: 0.9111 - loss: 0.2401 - val_accuracy: 0.9047 - val_loss: 0.2526
Epoch 3/3
307/439 ━━━━━━━━━━━━━━━━━━━━ 40s 306ms/step - accuracy: 0.9267 - loss: 0.1970